# Avant le TP8 : la boîte à outils du diagnostic

Au TP8, on ne vous dit pas ce qui est cassé : un ticket décrit un symptôme, et vous enquêtez. Cette démonstration montre, sur le lab en bon état, les quatre réflexes de la boîte à outils :

1. un **bilan** général, du quorum aux alertes, qui sert de référence ;
2. la **comparaison** d'une configuration avec sa version d'origine ;
3. la recherche dans les **traces** d'un nœud ;
4. les **alertes** et une métrique de Prometheus.

Chaque étape montre la version **terminal**, puis la version **Python**. Tout est en lecture : les deux versions sont exécutées. En terminal, les requêtes à Prometheus passent par `wget`, lancé dans le conteneur `prometheus`.

Sorties de référence : exécution du 12/11/2026 22:57 UTC sur le lab (mode `extended`).

Les adresses, les noms et les seuils sont définis une seule fois, ici.

In [1]:
import difflib
import os
import subprocess

import requests
from confluent_kafka import KafkaException
from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]
CONF = {"bootstrap.servers": BS}
LAB = os.environ["FORMATION_DIR"] + "/lab"
PROMETHEUS = "http://prometheus:9090"
DELAY = 10  # secondes

NODE = "kafka-1"  # nœud dont on compare la configuration et lit les traces
NODES = ("kafka-1", "kafka-2", "kafka-3")
WINDOW = "10m"  # fenêtre de recherche dans les traces
RATE_PER_TOPIC = 'sum by (topic) (rate(kafka_server_brokertopicmetrics_messagesin_total{topic!=""}[1m]))'

## 1. Le bilan de référence

Le bilan rassemble l'état général : le leader du quorum, le nom sous lequel chaque broker est annoncé aux clients, les partitions dégradées et les alertes. Exécuté sur un lab sain, il donne la référence « tout va bien ».

**En terminal** : une commande par indicateur. Les deux filtres de `kafka-topics.sh` n'affichent rien quand tout va bien.

In [2]:
%%bash
echo "--- quorum";        docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status | grep LeaderId
echo "--- brokers";       docker exec kafka-tools kafka-broker-api-versions.sh --bootstrap-server $BS | grep -oE '^[^ ]+ \(id: [0-9]+ rack: [^)]+\)' | sort
echo "--- urp";           docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --under-replicated-partitions
echo "--- indisponibles"; docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --unavailable-partitions
echo "--- alertes";       docker exec prometheus wget -qO- http://localhost:9090/api/v1/alerts

--- quorum


LeaderId:               1


--- brokers


kafka-1:19092 (id: 1 rack: az-a isFenced: false)
kafka-2:19092 (id: 2 rack: az-b isFenced: false)
ka

fka-3:19092 (id: 3 rack: az-c isFenced: false)


--- urp


--- indisponibles


--- alertes


{"status":"success","data":{"alerts":[]}}

**En Python** : une seule fonction, `summary()`, que vous reprendrez au TP en y ajoutant un indicateur. Le quorum n'a pas d'API Python : on lance l'outil Java et on lit sa sortie. Les brokers se lisent dans les métadonnées (`list_topics().brokers`), qui répondent même quand un nœud est injoignable.

In [3]:
def alerts():
    response = requests.get(PROMETHEUS + "/api/v1/alerts", timeout=DELAY)
    response.raise_for_status()
    return sorted({"%s:%s" % (a["labels"]["alertname"], a["state"]) for a in response.json()["data"]["alerts"]})


def quorum_leader():
    status = subprocess.run(["docker", "exec", "kafka-tools", "kafka-metadata-quorum.sh",
                             "--bootstrap-server", BS, "describe", "--status"],
                            capture_output=True, text=True, timeout=60)
    if status.returncode != 0 or "LeaderId:" not in status.stdout:
        return "inconnu (%s)" % status.stderr.strip().splitlines()[-1:]
    return status.stdout.split("LeaderId:")[1].split()[0]


def summary():
    admin = AdminClient(CONF)
    try:
        md = admin.list_topics(timeout=DELAY)
    except KafkaException as e:
        print("métadonnées indisponibles :", e.args[0].name())
        return
    partitions = [p for t in md.topics.values() for p in t.partitions.values()]
    print("quorum         leader", quorum_leader())
    print("brokers       ", {b.id: b.host for b in sorted(md.brokers.values(), key=lambda b: b.id)})
    print("urp           ", sum(1 for p in partitions if len(p.isrs) < len(p.replicas)))
    print("indisponibles ", sum(1 for p in partitions if p.leader == -1))
    try:
        print("alertes       ", alerts())
    except requests.RequestException as e:
        print("alertes        Prometheus injoignable :", e)


summary()

quorum         leader 1
brokers        {1: 'kafka-1', 2: 'kafka-2', 3: 'kafka-3'}
urp            0
indisponibles  0
alertes        []


## 2. Qu'est-ce qui a changé ?

Le lab garde la configuration d'origine de chaque nœud dans `config/kafka/pristine/`. Une différence avec le fichier en service répond à la question « qu'est-ce qui a changé ? ». Ici, `kafka-1` : s'il a gardé le second répertoire de données du TP7, la différence le montre ; sinon, il n'y a rien à afficher.

**En terminal** : `diff` rend le code 1 quand les fichiers diffèrent, 0 quand ils sont identiques.

In [4]:
%%bash
cd $FORMATION_DIR/lab
diff config/kafka/pristine/kafka-1.server.properties config/kafka/kafka-1/server.properties
echo "code retour de diff : $? (0 : identiques, 1 : différents)"

76c76
< log.dirs=/var/lib/kafka/data
---
> log.dirs=/var/lib/kafka/data,/var/lib/kafka/data2


code retour de diff : 1 (0 : identiques, 1 : différents)


**En Python** : `difflib` compare les deux fichiers ligne à ligne ; seules les lignes retirées (`-`) ou ajoutées (`+`) sont gardées.

In [5]:
with open(f"{LAB}/config/kafka/pristine/{NODE}.server.properties") as f:
    origin = f.read().splitlines()
with open(f"{LAB}/config/kafka/{NODE}/server.properties") as f:
    in_service = f.read().splitlines()

differences = [l for l in difflib.unified_diff(origin, in_service, lineterm="", n=0)
               if l[:1] in "+-" and not l.startswith(("+++", "---"))]
print(f"{NODE} :", differences or "identique à l'origine")

kafka-1 : ['-log.dirs=/var/lib/kafka/data', '+log.dirs=/var/lib/kafka/data,/var/lib/kafka/data2']


## 3. Chercher dans les traces d'un nœud

`docker compose logs` donne les traces d'un nœud ; `grep -E` les filtre. On compte d'abord les avertissements et les erreurs des 10 dernières minutes, puis on lit les dernières lignes.

**En terminal**

In [6]:
%%bash
cd $FORMATION_DIR/lab
for n in 1 2 3; do
  echo "kafka-$n : $(docker compose logs kafka-$n --since 10m --no-log-prefix 2>&1 | grep -cE ' (WARN|ERROR) ') ligne(s) WARN ou ERROR en 10 min"
done
docker compose logs kafka-1 --no-log-prefix 2>&1 | grep -E ' (INFO|WARN) ' | tail -3 | cut -c1-200

kafka-1 : 44 ligne(s) WARN ou ERROR en 10 min


kafka-2 : 66 ligne(s) WARN ou ERROR en 10 min


kafka-3 : 56 ligne(s) WARN ou ERROR en 10 min


[2026-11-12 22:57:42,493] INFO [DynamicConfigPublisher broker id=1] Updating topic orders with new c

onfiguration :  (kafka.server.metadata.DynamicConfigPublisher)
[2026-11-12 22:57:42,495] INFO [Dynam

icConfigPublisher broker id=1] Updating topic orders-history with new configuration :  (kafka.server

.metadata.DynamicConfigPublisher)
[2026-11-12 22:57:42,495] INFO [DynamicConfigPublisher broker id=1

] Updating topic payments with new configuration :  (kafka.server.metadata.DynamicConfigPublisher)


**En Python** : `subprocess.run()` lance la même commande et rend sa sortie ; le filtre et le compte se font en Python.

In [7]:
def logs(node, since=WINDOW):
    output = subprocess.run(["docker", "compose", "logs", node, "--since", since, "--no-log-prefix"],
                            cwd=LAB, capture_output=True, text=True, timeout=60)
    if output.returncode != 0:
        raise RuntimeError(output.stderr.strip())
    return output.stdout.splitlines()


for node in NODES:
    problems = [l for l in logs(node) if " WARN " in l or " ERROR " in l]
    print(f"{node} : {len(problems)} ligne(s) WARN ou ERROR en {WINDOW}")

kafka-1 : 44 ligne(s) WARN ou ERROR en 10m


kafka-2 : 66 ligne(s) WARN ou ERROR en 10m


kafka-3 : 56 ligne(s) WARN ou ERROR en 10m


## 4. Alertes et métriques

Les alertes disent **que** quelque chose ne va pas ; les métriques aident à trouver **où**. Exemple : le débit d'entrée par topic.

**En terminal** : la requête PromQL est envoyée par `--post-data`, ce qui évite d'encoder ses accolades et ses crochets dans l'URL.

In [8]:
%%bash
docker exec prometheus wget -qO- --post-data 'query=sum by (topic) (rate(kafka_server_brokertopicmetrics_messagesin_total{topic!=""}[1m]))' http://localhost:9090/api/v1/query

{"status":"success","data":{"resultType":"vector","result":[{"metric":{"topic":"__consumer_offsets"}

,"value":[1794524279.490,"1.2"]},{"metric":{"topic":"loadgen"},"value":[1794524279.490,"98.585276746

07237"]},{"metric":{"topic":"payments"},"value":[1794524279.490,"170.59506254748996"]},{"metric":{"t

opic":"orders-history"},"value":[1794524279.490,"0"]}]}}

**En Python**

In [9]:
response = requests.get(PROMETHEUS + "/api/v1/query", params={"query": RATE_PER_TOPIC}, timeout=DELAY)
response.raise_for_status()
for result in response.json()["data"]["result"]:
    print("%-20s %6.1f messages/s" % (result["metric"]["topic"], float(result["value"][1])))

__consumer_offsets      1.2 messages/s
loadgen                98.6 messages/s
payments              170.6 messages/s
orders-history          0.0 messages/s


## La méthode du TP8

Ouvrez `tps/jour2/tp8_incidents/enonce.ipynb`, et dupliquez `fiche-diagnostic.md` une fois par ticket.

- Qualifier, observer du général au particulier, formuler une hypothèse, la vérifier par une commande, corriger, valider. Une cellule par hypothèse.
- Observez avant d'agir : un redémarrage efface des indices.
- Votre partenaire injecte l'incident dans un terminal de votre poste ; il joue l'équipe qui a ouvert le ticket et répond sur le contexte, jamais sur la cause.